# Model 07 — Gaussian Naive Bayes

## BRFSS 2025 Diabetes Classification

**Purpose:** execute and freeze the seventh and final primary classifier without rerunning Models 01–06.

### Literature provenance

**BRFSS application:** Xie et al. (2019), *Preventing Chronic Disease / CDC*, DOI `10.5888/pcd16.190109`, directly evaluated Gaussian Naive Bayes on BRFSS 2014.

**Architecture reference:** Friedman, Geiger, & Goldszmidt (1997), *Bayesian Network Classifiers*, *Machine Learning*, DOI `10.1023/A:1007465528199`.

Gaussian Naive Bayes provides a computationally efficient probabilistic baseline with a very different inductive bias from the linear-margin and tree-ensemble models already evaluated.


## 1. Frozen Study Protocol

Inherited unchanged from Part II:

- target: `DIABETE4=1` vs `DIABETE4=3`;
- same 20 predictors;
- 80/20 stratified split with `random_state=42`;
- same missing-value handling;
- same one-hot categorical representation;
- same numeric scaling;
- Stratified 5-Fold CV on training data;
- no SMOTE or class balancing;
- untouched test set.

### Dense conversion

The frozen preprocessing pipeline outputs a sparse matrix because of one-hot encoding.

`GaussianNB` requires dense input, so Model 07 adds only:

```text
frozen Part II preprocessing
        ↓
sparse matrix → dense array
        ↓
GaussianNB
```

No values, categories, scaling rules, or predictors are changed.


In [ ]:
from pathlib import Path
from math import sqrt
import json
import platform
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "DIABETE4"

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

xpt_candidates = glob.glob("/kaggle/input/**/LLCP2025.XPT", recursive=True)
if len(xpt_candidates) != 1:
    raise RuntimeError(f"Expected exactly one LLCP2025.XPT input, found: {xpt_candidates}")
xpt_path = Path(xpt_candidates[0])

print("Input:", xpt_path)
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)

    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)

    return out[PRIMARY_FEATURES]

target_mask = df[TARGET].isin([1.0, 3.0])
X = recode_brfss_2025(df.loc[target_mask, PRIMARY_FEATURES].copy())
y = (df.loc[target_mask, TARGET] == 1.0).astype("int8").rename("diabetes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

assert len(X) == 342_539
assert len(X_train) == 274_031
assert len(X_test) == 68_508
assert int(y.sum()) == 51_827

print("Cohort:", len(X))
print("Train:", len(X_train), "Test:", len(X_test))


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

dense_transformer = FunctionTransformer(
    to_dense,
    accept_sparse=True,
    feature_names_out="one-to-one",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

specificity_scorer = make_scorer(recall_score, pos_label=0)
precision_scorer = make_scorer(precision_score, zero_division=0)
scoring = {
    "accuracy": "accuracy",
    "precision": precision_scorer,
    "recall": "recall",
    "specificity": specificity_scorer,
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}


## 2. Gaussian Naive Bayes Architecture

Naive Bayes applies Bayes' rule under the simplifying assumption that predictors are conditionally independent given the outcome class.

Gaussian Naive Bayes additionally models each feature's class-conditional distribution as Gaussian.

The fixed baseline is:

```python
GaussianNB(
    priors=None,
    var_smoothing=1e-9,
)
```

### Provenance classification

**Literature-supported**
- Gaussian Naive Bayes directly used on BRFSS diabetes in P02;
- probabilistic Naive Bayes classifier family;
- held-out evaluation.

**Study-specific**
- sklearn `GaussianNB`;
- default empirical class priors;
- `var_smoothing=1e-9`;
- sparse-to-dense adapter;
- no SMOTE;
- probability threshold 0.5.

### Assumption caveat

After one-hot encoding, many transformed features are binary rather than truly Gaussian. Model 07 is therefore treated as a simple probabilistic baseline, not as a claim that the Gaussian feature-distribution assumption is scientifically correct for every BRFSS indicator.


In [ ]:
MODEL_07_DIR = Path("/kaggle/working/model_07_gaussian_naive_bayes")
MODEL_07_DIR.mkdir(parents=True, exist_ok=True)

gnb = GaussianNB(
    priors=None,
    var_smoothing=1e-9,
)

gnb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("to_dense", dense_transformer),
    ("model", gnb),
])

config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "GaussianNB",
    "architecture": "Gaussian class-conditional naive Bayes classifier",
    "priors": None,
    "var_smoothing": 1e-9,
    "dense_adapter": True,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}

with open(MODEL_07_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


## 3. Five-Fold Cross-Validation

The complete preprocessing + dense conversion + GaussianNB pipeline is re-fit independently inside each training fold.

Dense conversion is representation-only: it does not alter feature values.


In [ ]:
gnb_cv_raw = cross_validate(
    gnb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(gnb_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(gnb_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(gnb_cv_raw["score_time"][fold])
    fold_rows.append(row)

gnb_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = gnb_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

gnb_cv_summary = pd.DataFrame(summary_rows)
display(gnb_cv_folds)
display(gnb_cv_summary)

gnb_cv_folds.to_csv(MODEL_07_DIR / "cv_folds.csv", index=False)
gnb_cv_summary.to_csv(MODEL_07_DIR / "cv_summary.csv", index=False)


## 4. Fit Frozen Model and Evaluate the Untouched Test Set

The exact same fixed model is fit once on all training respondents.

The test set is evaluated once at probability threshold 0.5. No prior, smoothing, or threshold value is changed afterward.


In [ ]:
gnb_pipeline.fit(X_train, y_train)

y_test_prob_gnb = gnb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_gnb = (y_test_prob_gnb >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_gnb, labels=[0, 1]).ravel()

test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_gnb)),
    "precision": float(precision_score(y_test, y_test_pred_gnb, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_gnb)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_gnb)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_gnb)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_gnb)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_gnb)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "threshold": 0.5,
}

print(json.dumps(test_metrics, indent=2))

with open(MODEL_07_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)


## 5. Stratified Bootstrap Confidence Intervals

Two hundred class-stratified bootstrap samples quantify uncertainty around the fixed held-out predictions.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251009

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_gnb[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()
    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc","brier_score"]:
    values = bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

test_summary = pd.DataFrame(test_summary_rows)
display(test_summary)

bootstrap_df.to_csv(MODEL_07_DIR / "bootstrap_metrics.csv", index=False)
test_summary.to_csv(MODEL_07_DIR / "test_summary.csv", index=False)


## 6. Diagnostics

ROC, precision-recall, confusion matrix, and class-conditional Gaussian parameters are saved.

The learned means/variances are descriptive model parameters only and are not causal effects.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_gnb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_gnb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Gaussian NB (AUC={test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 07 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Gaussian NB (AP={test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 07 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 07 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = gnb_pipeline.named_steps["preprocess"].get_feature_names_out()
model = gnb_pipeline.named_steps["model"]
params = pd.DataFrame({
    "transformed_feature": feature_names,
    "mean_no_diabetes": model.theta_[0],
    "mean_diabetes": model.theta_[1],
    "variance_no_diabetes": model.var_[0],
    "variance_diabetes": model.var_[1],
})
params.to_csv(MODEL_07_DIR / "gaussian_parameters.csv", index=False)
display(params.head(20))


## 7. Freeze Rule

After verified execution, Model 07 is frozen as:

> **Gaussian Naive Bayes with empirical class priors, var_smoothing=1e-9, dense adapter after frozen preprocessing, no resampling, threshold=0.5.**

No post-test changes are allowed to priors, smoothing, threshold, preprocessing, or feature set.

This completes the **seven-model primary benchmark**. Any KNN, neural network, calibrated model, tuned model, or resampling variant belongs to sensitivity/secondary experiments rather than the primary comparison.
